# A 2-D U-Net forward pass

Build a three-class network and inspect its output on the real `r16` image from
the [ANTsX tutorial](https://gist.github.com/ntustison/12a656a5fc2f6f9c4494c88dc09c5621). The input is resized to 64 × 64 and normalized; it is not random
noise. The network still has random weights and is not a trained anatomical
segmentation model. ANTsPy downloads the sample on first use.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

## Construct a small network and load the tutorial image

In [ ]:
torch.manual_seed(42)
model = antstorch.create_unet_model_2d(
    input_channel_size=1, number_of_outputs=3,
    number_of_filters=(8, 16, 32), mode="classification",
).to(device).eval()
sample = ants.image_read(ants.get_ants_data("r16")).clone("float")
sample = ants.resample_image(sample, (64, 64), use_voxels=True)
sample = ants.iMath(sample, "Normalize")
image = torch.from_numpy(sample.numpy().T.copy())[None, None].to(device)
with torch.inference_mode():
    probabilities = model(image)
print("Output layout (batch, classes, height, width):", tuple(probabilities.shape))

## Check the output contract
Classification mode returns a probability per class and voxel. The class
probabilities should be finite, lie in [0, 1], and sum to one.

In [ ]:
assert probabilities.shape == (1, 3, 64, 64)
assert torch.isfinite(probabilities).all()
assert ((probabilities >= 0) & (probabilities <= 1)).all()
torch.testing.assert_close(probabilities.sum(dim=1), torch.ones_like(image[:, 0]))
fig, axes = plt.subplots(1, 4, figsize=(12, 3), constrained_layout=True)
axes[0].imshow(image[0, 0].cpu(), cmap="gray")
axes[0].set_title("Tutorial r16")
for index in range(3):
    axes[index + 1].imshow(probabilities[0, index].cpu(), vmin=0, vmax=1)
    axes[index + 1].set_title(f"Class {index} probability")
for ax in axes: ax.axis("off")
plt.show()

Train a model on suitable labeled images before interpreting its predictions.